In [1]:
from pathlib import Path
import getpass
import warnings
import re

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_PATH = Path(r"C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM")

PROCESSED_DATA_PATH = PROJECT_PATH / "data" / "processed"

CLV_OUTPUT_PATH = PROCESSED_DATA_PATH / "customer_lifetime_value"

CLV_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "ecommerce_ai_db"

SOURCE_SCHEMA = "analytics"
OUTPUT_SCHEMA = "analytics"

print(f"Project path: {PROJECT_PATH}")
print(f"CLV output path: {CLV_OUTPUT_PATH}")
print(f"Database: {DB_NAME}")
print(f"Source schema: {SOURCE_SCHEMA}")
print(f"Output schema: {OUTPUT_SCHEMA}")

Project path: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM
CLV output path: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\customer_lifetime_value
Database: ecommerce_ai_db
Source schema: analytics
Output schema: analytics


In [3]:
if not PROJECT_PATH.exists():
    raise FileNotFoundError(
        f"Project path does not exist:\n{PROJECT_PATH}"
    )

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed data path does not exist:\n{PROCESSED_DATA_PATH}"
    )

print("Project and processed-data paths verified successfully.")

Project and processed-data paths verified successfully.


In [4]:
DB_USER = input("Enter PostgreSQL username: ").strip()

if not DB_USER:
    raise ValueError("PostgreSQL username cannot be empty.")

DB_PASSWORD = getpass.getpass("Enter PostgreSQL password: ")

if not DB_PASSWORD:
    raise ValueError("PostgreSQL password cannot be empty.")

print("Database credentials received securely.")

Enter PostgreSQL username:  postgres
Enter PostgreSQL password:  ········


Database credentials received securely.


In [5]:
DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    future=True
)

print("SQLAlchemy engine created successfully.")

SQLAlchemy engine created successfully.


In [6]:
try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    
    print("PostgreSQL connection verified successfully.")

except SQLAlchemyError as error:
    raise ConnectionError(
        "PostgreSQL connection failed. "
        "Verify PostgreSQL is running and that the host, port, database, "
        "username, and password are correct."
    ) from error

PostgreSQL connection verified successfully.


In [7]:
with engine.begin() as connection:
    connection.execute(
        text(
            f"""
            CREATE SCHEMA IF NOT EXISTS "{OUTPUT_SCHEMA}";
            """
        )
    )

print(f"Schema verified: {OUTPUT_SCHEMA}")

Schema verified: analytics


In [8]:
schema_query = text(
    """
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name = :schema_name;
    """
)

schema_check_df = pd.read_sql(
    schema_query,
    engine,
    params={"schema_name": SOURCE_SCHEMA}
)

if schema_check_df.empty:
    raise RuntimeError(
        f"The required PostgreSQL schema '{SOURCE_SCHEMA}' does not exist."
    )

print(f"Verified PostgreSQL schema: {SOURCE_SCHEMA}")

Verified PostgreSQL schema: analytics


In [9]:
tables_query = text(
    """
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema = :schema_name
    ORDER BY table_name;
    """
)

analytics_tables_df = pd.read_sql(
    tables_query,
    engine,
    params={"schema_name": SOURCE_SCHEMA}
)

if analytics_tables_df.empty:
    raise RuntimeError(
        f"No tables were found in the '{SOURCE_SCHEMA}' schema."
    )

display(analytics_tables_df)

,table_schema,table_name,table_type
0,analytics,customer_segmentation,BASE TABLE
1,analytics,customer_segmentation_model_evaluation,BASE TABLE
2,analytics,segment_distribution,BASE TABLE
3,analytics,segment_feature_means,BASE TABLE
4,analytics,segment_summary,BASE TABLE


In [10]:
columns_query = text(
    """
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        udt_name,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema = :schema_name
    ORDER BY table_name, ordinal_position;
    """
)

analytics_columns_df = pd.read_sql(
    columns_query,
    engine,
    params={"schema_name": SOURCE_SCHEMA}
)

if analytics_columns_df.empty:
    raise RuntimeError(
        f"No columns were found in the '{SOURCE_SCHEMA}' schema."
    )

display(analytics_columns_df)

,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable
0,analytics,customer_segmentation,1,customer_id,text,text,YES
1,analytics,customer_segmentation,2,customer_zip_code_prefix,double precision,float8,YES
2,analytics,customer_segmentation,3,total_orders,double precision,float8,YES
3,analytics,customer_segmentation,4,total_spend,double precision,float8,YES
4,analytics,customer_segmentation,5,average_order_value,double precision,float8,YES
5,analytics,customer_segmentation,6,total_items_purchased,double precision,float8,YES
6,analytics,customer_segmentation,7,average_review_score,double precision,float8,YES
7,analytics,customer_segmentation,8,average_delivery_days,double precision,float8,YES
8,analytics,customer_segmentation,9,average_delivery_delay_days,double precision,float8,YES
9,analytics,customer_segmentation,10,late_orders,double precision,float8,YES


In [11]:
segmentation_table_candidates = [
    "customer_segmentation",
    "customer_segments",
    "customer_segment"
]

available_table_names = set(
    analytics_tables_df["table_name"]
    .astype(str)
    .str.lower()
)

segmentation_table_name = None

for candidate in segmentation_table_candidates:
    if candidate.lower() in available_table_names:
        segmentation_table_name = candidate
        break

if segmentation_table_name is None:
    segmentation_like_tables = [
        table_name
        for table_name in analytics_tables_df["table_name"].astype(str)
        if "segment" in table_name.lower()
    ]

    if len(segmentation_like_tables) == 1:
        segmentation_table_name = segmentation_like_tables[0]

if segmentation_table_name is None:
    raise RuntimeError(
        "Could not identify a valid customer segmentation table in "
        f"the '{SOURCE_SCHEMA}' schema."
    )

print(f"Verified segmentation table: {segmentation_table_name}")

Verified segmentation table: customer_segmentation


In [12]:
segmentation_columns_df = analytics_columns_df[
    analytics_columns_df["table_name"].astype(str).str.lower()
    == segmentation_table_name.lower()
].copy()

if segmentation_columns_df.empty:
    raise RuntimeError(
        f"No metadata was found for table '{segmentation_table_name}'."
    )

display(segmentation_columns_df)

,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable
0,analytics,customer_segmentation,1,customer_id,text,text,YES
1,analytics,customer_segmentation,2,customer_zip_code_prefix,double precision,float8,YES
2,analytics,customer_segmentation,3,total_orders,double precision,float8,YES
3,analytics,customer_segmentation,4,total_spend,double precision,float8,YES
4,analytics,customer_segmentation,5,average_order_value,double precision,float8,YES
5,analytics,customer_segmentation,6,total_items_purchased,double precision,float8,YES
6,analytics,customer_segmentation,7,average_review_score,double precision,float8,YES
7,analytics,customer_segmentation,8,average_delivery_days,double precision,float8,YES
8,analytics,customer_segmentation,9,average_delivery_delay_days,double precision,float8,YES
9,analytics,customer_segmentation,10,late_orders,double precision,float8,YES


In [13]:
segmentation_column_names = (
    segmentation_columns_df["column_name"]
    .astype(str)
    .tolist()
)

segmentation_column_names_lower = {
    column.lower(): column
    for column in segmentation_column_names
}

print("Verified segmentation columns:")
for column in segmentation_column_names:
    print(f"- {column}")

Verified segmentation columns:
- customer_id
- customer_zip_code_prefix
- total_orders
- total_spend
- average_order_value
- total_items_purchased
- average_review_score
- average_delivery_days
- average_delivery_delay_days
- late_orders
- cancelled_orders
- customer_lifetime_days
- purchase_frequency
- recency_days
- frequency
- monetary
- cluster_id
- segment_name


In [14]:
identifier_priority_patterns = [
    r"^customer_unique_id$",
    r"^customer_id$",
    r"^user_id$",
    r"^client_id$",
    r".*customer.*id.*",
    r".*user.*id.*",
    r".*client.*id.*",
    r".*identifier.*"
]

customer_identifier_column = None

for pattern in identifier_priority_patterns:
    matching_columns = [
        column
        for column in segmentation_column_names
        if re.fullmatch(pattern, column, flags=re.IGNORECASE)
    ]

    if matching_columns:
        customer_identifier_column = matching_columns[0]
        break

if customer_identifier_column is None:
    raise RuntimeError(
        "Could not safely identify a customer identifier in the "
        f"'{segmentation_table_name}' table."
    )

identifier_metadata = segmentation_columns_df[
    segmentation_columns_df["column_name"]
    == customer_identifier_column
].iloc[0]

print(f"Verified customer identifier: {customer_identifier_column}")
print(f"PostgreSQL data type: {identifier_metadata['data_type']}")
print(f"PostgreSQL UDT: {identifier_metadata['udt_name']}")

Verified customer identifier: customer_id
PostgreSQL data type: text
PostgreSQL UDT: text


In [15]:
numeric_data_types = {
    "smallint",
    "integer",
    "bigint",
    "decimal",
    "numeric",
    "real",
    "double precision",
    "smallserial",
    "serial",
    "bigserial",
    "money"
}

numeric_feature_metadata_df = segmentation_columns_df[
    segmentation_columns_df["data_type"]
    .astype(str)
    .str.lower()
    .isin(numeric_data_types)
].copy()

numeric_feature_columns = [
    column
    for column in numeric_feature_metadata_df["column_name"].astype(str)
    if column != customer_identifier_column
]

if not numeric_feature_columns:
    raise RuntimeError(
        "No verified numeric customer-level features were found in "
        f"'{segmentation_table_name}'."
    )

print("Verified numeric customer-level features:")
for column in numeric_feature_columns:
    print(f"- {column}")

Verified numeric customer-level features:
- customer_zip_code_prefix
- total_orders
- total_spend
- average_order_value
- total_items_purchased
- average_review_score
- average_delivery_days
- average_delivery_delay_days
- late_orders
- cancelled_orders
- customer_lifetime_days
- purchase_frequency
- recency_days
- frequency
- monetary
- cluster_id


In [16]:
monetary_priority_patterns = [
    r".*total.*revenue.*",
    r".*total.*sales.*",
    r".*total.*spend.*",
    r".*total.*amount.*",
    r".*customer.*value.*",
    r".*monetary.*",
    r".*revenue.*",
    r".*sales.*",
    r".*spend.*",
    r".*amount.*",
    r".*payment.*"
]

monetary_feature_column = None

for pattern in monetary_priority_patterns:
    matching_columns = [
        column
        for column in numeric_feature_columns
        if re.fullmatch(pattern, column, flags=re.IGNORECASE)
    ]

    if matching_columns:
        monetary_feature_column = matching_columns[0]
        break

if monetary_feature_column is not None:
    monetary_metadata = numeric_feature_metadata_df[
        numeric_feature_metadata_df["column_name"]
        == monetary_feature_column
    ].iloc[0]

    print(f"Verified monetary/customer-value feature: {monetary_feature_column}")
    print(f"PostgreSQL data type: {monetary_metadata['data_type']}")
    print(f"PostgreSQL UDT: {monetary_metadata['udt_name']}")

else:
    print(
        "No monetary/customer-value feature was identified safely "
        "from the segmentation table."
    )

Verified monetary/customer-value feature: total_spend
PostgreSQL data type: double precision
PostgreSQL UDT: float8


In [17]:
segmentation_table_identifier = (
    f'"{SOURCE_SCHEMA}"."{segmentation_table_name}"'
)

customer_identifier_sql = (
    f'"{customer_identifier_column}"'
)

segmentation_data_query = text(
    f"""
    SELECT *
    FROM {segmentation_table_identifier};
    """
)

segmentation_df = pd.read_sql(
    segmentation_data_query,
    engine
)

if segmentation_df.empty:
    raise RuntimeError(
        f"The table {segmentation_table_identifier} contains no rows."
    )

print(f"Loaded rows: {len(segmentation_df):,}")
print(f"Loaded columns: {len(segmentation_df.columns):,}")

display(segmentation_df.head())

Loaded rows: 99,441
Loaded columns: 18


,customer_id,customer_zip_code_prefix,total_orders,total_spend,average_order_value,total_items_purchased,average_review_score,average_delivery_days,average_delivery_delay_days,late_orders,cancelled_orders,customer_lifetime_days,purchase_frequency,recency_days,frequency,monetary,cluster_id,segment_name
0,000419c5494106c306a97b5635748086,24220.0,1.0,49.40,49.40,1.0,1.0,45.979097,26.720532,1.0,0.0,0.0,1.0,229.987940,1.0,49.40,1,Higher Activity Segment
1,0017a0b4c1f1bdb9c395fa0ac517109c,81510.0,1.0,50.01,50.01,1.0,4.0,20.149248,1.929815,1.0,0.0,0.0,1.0,228.948808,1.0,50.01,1,Higher Activity Segment
2,001df1ee5c36767aa607001ab1a13a06,1030.0,1.0,42.86,42.86,1.0,4.0,4.818044,0.786620,1.0,0.0,0.0,1.0,73.760799,1.0,42.86,1,Higher Activity Segment
3,002b5342c72978cf0aba6aae1f5d5293,22775.0,1.0,0.00,0.00,0.0,1.0,9.255961,0.000000,0.0,1.0,0.0,1.0,37.985336,1.0,0.00,1,Higher Activity Segment
4,003f7d92ac63c512bb6584219806f8df,4310.0,1.0,84.91,84.91,1.0,1.0,28.429178,8.960648,1.0,0.0,0.0,1.0,246.197905,1.0,84.91,1,Higher Activity Segment


In [18]:
if customer_identifier_column not in segmentation_df.columns:
    raise RuntimeError(
        f"Verified identifier '{customer_identifier_column}' "
        "was not found in the loaded segmentation data."
    )

segmentation_df[customer_identifier_column] = (
    segmentation_df[customer_identifier_column]
    .astype("string")
    .str.strip()
)

segmentation_df = segmentation_df[
    segmentation_df[customer_identifier_column].notna()
    & (
        segmentation_df[customer_identifier_column]
        != ""
    )
].copy()

if segmentation_df.empty:
    raise RuntimeError(
        "No valid customer identifiers remain after cleaning."
    )

duplicate_customer_count = (
    segmentation_df[customer_identifier_column]
    .duplicated()
    .sum()
)

print(f"Duplicate customer rows detected: {duplicate_customer_count:,}")

Duplicate customer rows detected: 0


In [19]:
if duplicate_customer_count > 0:
    numeric_aggregation_map = {
        column: "mean"
        for column in numeric_feature_columns
        if column in segmentation_df.columns
    }

    non_numeric_columns = [
        column
        for column in segmentation_df.columns
        if column != customer_identifier_column
        and column not in numeric_aggregation_map
    ]

    duplicate_aggregation_map = numeric_aggregation_map.copy()

    for column in non_numeric_columns:
        duplicate_aggregation_map[column] = "first"

    segmentation_df = (
        segmentation_df
        .groupby(
            customer_identifier_column,
            as_index=False,
            dropna=False
        )
        .agg(duplicate_aggregation_map)
    )

print(
    "Customer-level uniqueness verified:",
    segmentation_df[customer_identifier_column].is_unique
)

Customer-level uniqueness verified: True


In [20]:
verified_numeric_columns = [
    column
    for column in numeric_feature_columns
    if column in segmentation_df.columns
]

if not verified_numeric_columns:
    raise RuntimeError(
        "No verified numeric features are available in the loaded data."
    )

for column in verified_numeric_columns:
    segmentation_df[column] = pd.to_numeric(
        segmentation_df[column],
        errors="coerce"
    )

segmentation_df[verified_numeric_columns] = (
    segmentation_df[verified_numeric_columns]
    .replace([np.inf, -np.inf], np.nan)
)

numeric_quality_df = pd.DataFrame({
    "column_name": verified_numeric_columns,
    "missing_values": [
        int(segmentation_df[column].isna().sum())
        for column in verified_numeric_columns
    ],
    "infinite_values": [
        int(
            np.isinf(
                segmentation_df[column].fillna(0).to_numpy(
                    dtype=float
                )
            ).sum()
        )
        for column in verified_numeric_columns
    ],
    "valid_numeric_values": [
        int(segmentation_df[column].notna().sum())
        for column in verified_numeric_columns
    ]
})

display(numeric_quality_df)

,column_name,missing_values,infinite_values,valid_numeric_values
0,customer_zip_code_prefix,0,0,99441
1,total_orders,0,0,99441
2,total_spend,0,0,99441
3,average_order_value,0,0,99441
4,total_items_purchased,0,0,99441
5,average_review_score,0,0,99441
6,average_delivery_days,0,0,99441
7,average_delivery_delay_days,0,0,99441
8,late_orders,0,0,99441
9,cancelled_orders,0,0,99441


In [21]:
if monetary_feature_column is not None:
    monetary_feature_column = (
        monetary_feature_column
        if monetary_feature_column in segmentation_df.columns
        else None
    )

if monetary_feature_column is None:
    raise RuntimeError(
        "A verified monetary/customer-value feature could not be safely "
        "identified from the customer segmentation output.\n\n"
        "No invented monetary feature will be used.\n"
        "To continue, the previous feature-engineering pipeline must expose "
        "a verified customer-level monetary/value feature."
    )

monetary_values = pd.to_numeric(
    segmentation_df[monetary_feature_column],
    errors="coerce"
)

monetary_values = monetary_values.replace(
    [np.inf, -np.inf],
    np.nan
)

valid_monetary_values = monetary_values[
    monetary_values.notna()
]

if valid_monetary_values.empty:
    raise RuntimeError(
        f"The verified monetary feature '{monetary_feature_column}' "
        "contains no valid numeric values."
    )

print(f"Using verified monetary feature: {monetary_feature_column}")
print(f"Valid monetary values: {len(valid_monetary_values):,}")
print(f"Minimum value: {valid_monetary_values.min():,.2f}")
print(f"Maximum value: {valid_monetary_values.max():,.2f}")

Using verified monetary feature: total_spend
Valid monetary values: 99,441
Minimum value: 0.00
Maximum value: 13,664.08


In [22]:
segmentation_df["historical_customer_value"] = (
    monetary_values
    .clip(lower=0)
)

segmentation_df["historical_customer_value"] = (
    segmentation_df["historical_customer_value"]
    .replace([np.inf, -np.inf], np.nan)
)

segmentation_df["historical_customer_value"] = (
    segmentation_df["historical_customer_value"]
    .fillna(0)
)

print(
    "Historical customer value created from verified feature:",
    monetary_feature_column
)

display(
    segmentation_df[
        [
            customer_identifier_column,
            "historical_customer_value"
        ]
    ].head()
)

Historical customer value created from verified feature: total_spend


,customer_id,historical_customer_value
0,000419c5494106c306a97b5635748086,49.40
1,0017a0b4c1f1bdb9c395fa0ac517109c,50.01
2,001df1ee5c36767aa607001ab1a13a06,42.86
3,002b5342c72978cf0aba6aae1f5d5293,0.00
4,003f7d92ac63c512bb6584219806f8df,84.91


In [23]:
historical_value_series = (
    segmentation_df["historical_customer_value"]
    .astype(float)
)

value_quantiles = (
    historical_value_series
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .quantile([0.01, 0.25, 0.50, 0.75, 0.99])
)

display(
    value_quantiles
    .rename("historical_customer_value")
    .to_frame()
)

,historical_customer_value
0.01,18.190
0.25,61.050
0.50,104.560
0.75,176.000
0.99,1057.688


In [24]:
if historical_value_series.notna().sum() == 0:
    raise RuntimeError(
        "No valid historical customer values are available."
    )

if (historical_value_series < 0).any():
    raise RuntimeError(
        "Negative historical customer values remain after validation."
    )

if not np.isfinite(
    historical_value_series.to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Infinite historical customer values remain after validation."
    )

print("Historical customer value validation passed.")

Historical customer value validation passed.


In [25]:
duration_priority_patterns = [
    r".*lifetime.*days.*",
    r".*customer.*lifetime.*",
    r".*active.*days.*",
    r".*tenure.*days.*",
    r".*duration.*days.*",
    r".*history.*days.*",
    r".*lifetime.*duration.*",
    r".*tenure.*"
]

duration_feature_column = None

for pattern in duration_priority_patterns:
    matching_columns = [
        column
        for column in verified_numeric_columns
        if re.fullmatch(pattern, column, flags=re.IGNORECASE)
    ]

    if matching_columns:
        duration_feature_column = matching_columns[0]
        break

if duration_feature_column is not None:
    print(
        "Verified customer lifetime-duration feature:",
        duration_feature_column
    )

else:
    print(
        "No verified customer lifetime-duration feature was found."
    )

Verified customer lifetime-duration feature: customer_lifetime_days


In [26]:
if duration_feature_column is not None:
    duration_values = pd.to_numeric(
        segmentation_df[duration_feature_column],
        errors="coerce"
    )

    duration_values = duration_values.replace(
        [np.inf, -np.inf],
        np.nan
    )

    duration_values = duration_values.clip(lower=0)

    segmentation_df["customer_lifetime_duration"] = (
        duration_values
    )

else:
    segmentation_df["customer_lifetime_duration"] = np.nan

print(
    "Customer lifetime duration column created:",
    "customer_lifetime_duration"
)

Customer lifetime duration column created: customer_lifetime_duration


In [27]:
segmentation_df["historical_clv"] = (
    segmentation_df["historical_customer_value"]
    .astype(float)
)

segmentation_df["historical_clv"] = (
    segmentation_df["historical_clv"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .clip(lower=0)
)

segmentation_df["estimated_future_clv"] = np.nan

print(
    "Historical CLV created successfully."
)

print(
    "Estimated future CLV intentionally remains unavailable because "
    "no validated future-retention model has been created."
)

Historical CLV created successfully.
Estimated future CLV intentionally remains unavailable because no validated future-retention model has been created.


In [28]:
clv_values = (
    segmentation_df["historical_clv"]
    .astype(float)
)

clv_values = clv_values.replace(
    [np.inf, -np.inf],
    np.nan
)

if clv_values.isna().any():
    raise RuntimeError(
        "Historical CLV contains invalid missing or infinite values."
    )

if (clv_values < 0).any():
    raise RuntimeError(
        "Historical CLV contains negative values."
    )

print("Historical CLV numerical validation passed.")

Historical CLV numerical validation passed.


In [29]:
clv_percentiles = (
    segmentation_df["historical_clv"]
    .quantile([0.25, 0.50, 0.75])
)

q25 = float(clv_percentiles.loc[0.25])
q50 = float(clv_percentiles.loc[0.50])
q75 = float(clv_percentiles.loc[0.75])

if q25 == q75:
    segmentation_df["clv_segment"] = np.select(
        [
            segmentation_df["historical_clv"] <= q50,
            segmentation_df["historical_clv"] > q50
        ],
        [
            "Low",
            "High"
        ],
        default="Medium"
    )

else:
    segmentation_df["clv_segment"] = pd.cut(
        segmentation_df["historical_clv"],
        bins=[
            -np.inf,
            q25,
            q75,
            np.inf
        ],
        labels=[
            "Low",
            "Medium",
            "High"
        ],
        include_lowest=True
    )

segmentation_df["clv_segment"] = (
    segmentation_df["clv_segment"]
    .astype("string")
)

print(
    segmentation_df["clv_segment"]
    .value_counts(dropna=False)
)

clv_segment
Medium    49707
Low       24874
High      24860
Name: count, dtype: Int64


In [30]:
valid_clv_segments = {
    "Low",
    "Medium",
    "High"
}

actual_clv_segments = set(
    segmentation_df["clv_segment"]
    .dropna()
    .astype(str)
    .unique()
)

if not actual_clv_segments.issubset(valid_clv_segments):
    raise RuntimeError(
        "Invalid CLV segment labels were generated."
    )

if segmentation_df["clv_segment"].isna().any():
    raise RuntimeError(
        "Missing CLV segment labels were generated."
    )

print("CLV segment validation passed.")

CLV segment validation passed.


In [31]:
clv_customer_output_columns = [
    customer_identifier_column,
    "historical_customer_value",
    "customer_lifetime_duration",
    "historical_clv",
    "estimated_future_clv",
    "clv_segment"
]

clv_customer_df = (
    segmentation_df[
        clv_customer_output_columns
    ]
    .copy()
)

if not clv_customer_df[
    customer_identifier_column
].is_unique:
    raise RuntimeError(
        "Customer-level CLV output contains duplicate customers."
    )

display(clv_customer_df.head())

,customer_id,historical_customer_value,customer_lifetime_duration,historical_clv,estimated_future_clv,clv_segment
0,000419c5494106c306a97b5635748086,49.40,0.0,49.40,NaN,Low
1,0017a0b4c1f1bdb9c395fa0ac517109c,50.01,0.0,50.01,NaN,Low
2,001df1ee5c36767aa607001ab1a13a06,42.86,0.0,42.86,NaN,Low
3,002b5342c72978cf0aba6aae1f5d5293,0.00,0.0,0.00,NaN,Low
4,003f7d92ac63c512bb6584219806f8df,84.91,0.0,84.91,NaN,Medium


In [32]:
clv_summary_df = (
    clv_customer_df
    .groupby(
        "clv_segment",
        dropna=False
    )
    .agg(
        customer_count=(
            customer_identifier_column,
            "nunique"
        ),
        total_historical_clv=(
            "historical_clv",
            "sum"
        ),
        average_historical_clv=(
            "historical_clv",
            "mean"
        ),
        median_historical_clv=(
            "historical_clv",
            "median"
        ),
        minimum_historical_clv=(
            "historical_clv",
            "min"
        ),
        maximum_historical_clv=(
            "historical_clv",
            "max"
        )
    )
    .reset_index()
)

total_clv = (
    clv_summary_df["total_historical_clv"]
    .sum()
)

if total_clv > 0:
    clv_summary_df["clv_value_share"] = (
        clv_summary_df["total_historical_clv"]
        / total_clv
    )

else:
    clv_summary_df["clv_value_share"] = 0.0

clv_summary_df["customer_share"] = (
    clv_summary_df["customer_count"]
    / clv_summary_df["customer_count"].sum()
)

display(clv_summary_df)

,clv_segment,customer_count,total_historical_clv,average_historical_clv,median_historical_clv,minimum_historical_clv,maximum_historical_clv,clv_value_share,customer_share
0,High,24860,9416402.07,378.777235,264.23,176.01,13664.08,0.594337,0.249997
1,Low,24874,1024405.69,41.183794,42.58,0.00,61.05,0.064658,0.250138
2,Medium,49707,5402745.48,108.691844,104.56,61.06,176.00,0.341006,0.499864


In [33]:
clv_summary_df["customer_count"] = (
    pd.to_numeric(
        clv_summary_df["customer_count"],
        errors="coerce"
    )
    .fillna(0)
    .astype("int64")
)

summary_numeric_columns = [
    "total_historical_clv",
    "average_historical_clv",
    "median_historical_clv",
    "minimum_historical_clv",
    "maximum_historical_clv",
    "clv_value_share",
    "customer_share"
]

for column in summary_numeric_columns:
    clv_summary_df[column] = (
        pd.to_numeric(
            clv_summary_df[column],
            errors="coerce"
        )
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

if not np.isclose(
    clv_summary_df["customer_share"].sum(),
    1.0,
    atol=1e-6
):
    raise RuntimeError(
        "Customer shares do not sum to approximately 1."
    )

if total_clv > 0:
    if not np.isclose(
        clv_summary_df["clv_value_share"].sum(),
        1.0,
        atol=1e-6
    ):
        raise RuntimeError(
            "CLV value shares do not sum to approximately 1."
        )

print("CLV summary validation passed.")

CLV summary validation passed.


In [34]:
clv_overall_summary_df = pd.DataFrame({
    "metric": [
        "customer_count",
        "total_historical_clv",
        "average_historical_clv",
        "median_historical_clv",
        "minimum_historical_clv",
        "maximum_historical_clv",
        "monetary_feature_used",
        "duration_feature_used",
        "clv_methodology"
    ],
    "value": [
        int(clv_customer_df[customer_identifier_column].nunique()),
        float(clv_customer_df["historical_clv"].sum()),
        float(clv_customer_df["historical_clv"].mean()),
        float(clv_customer_df["historical_clv"].median()),
        float(clv_customer_df["historical_clv"].min()),
        float(clv_customer_df["historical_clv"].max()),
        monetary_feature_column,
        duration_feature_column
        if duration_feature_column is not None
        else "Not available",
        "Historical observed customer value"
    ]
})

display(clv_overall_summary_df)

,metric,value
0,customer_count,99441
1,total_historical_clv,15843553.24
2,average_historical_clv,159.326166
3,median_historical_clv,104.56
4,minimum_historical_clv,0.0
5,maximum_historical_clv,13664.08
6,monetary_feature_used,total_spend
7,duration_feature_used,customer_lifetime_days
8,clv_methodology,Historical observed customer value


In [35]:
clv_validation_df = pd.DataFrame({
    "validation_check": [
        "customer_identifier_exists",
        "customer_identifier_unique",
        "customer_identifier_missing_count",
        "historical_clv_missing_count",
        "historical_clv_infinite_count",
        "historical_clv_negative_count",
        "clv_segment_missing_count",
        "valid_clv_segment_labels",
        "customer_count"
    ],
    "validation_value": [
        customer_identifier_column in clv_customer_df.columns,
        clv_customer_df[
            customer_identifier_column
        ].is_unique,
        int(
            clv_customer_df[
                customer_identifier_column
            ].isna().sum()
        ),
        int(
            clv_customer_df[
                "historical_clv"
            ].isna().sum()
        ),
        int(
            np.isinf(
                clv_customer_df[
                    "historical_clv"
                ].to_numpy(dtype=float)
            ).sum()
        ),
        int(
            (
                clv_customer_df[
                    "historical_clv"
                ] < 0
            ).sum()
        ),
        int(
            clv_customer_df[
                "clv_segment"
            ].isna().sum()
        ),
        actual_clv_segments.issubset(
            valid_clv_segments
        ),
        int(
            clv_customer_df[
                customer_identifier_column
            ].nunique()
        )
    ]
})

display(clv_validation_df)

,validation_check,validation_value
0,customer_identifier_exists,True
1,customer_identifier_unique,True
2,customer_identifier_missing_count,0
3,historical_clv_missing_count,0
4,historical_clv_infinite_count,0
5,historical_clv_negative_count,0
6,clv_segment_missing_count,0
7,valid_clv_segment_labels,True
8,customer_count,99441


In [36]:
if not bool(
    clv_validation_df.loc[
        clv_validation_df["validation_check"]
        == "customer_identifier_exists",
        "validation_value"
    ].iloc[0]
):
    raise AssertionError(
        "Customer identifier validation failed."
    )

if not bool(
    clv_validation_df.loc[
        clv_validation_df["validation_check"]
        == "customer_identifier_unique",
        "validation_value"
    ].iloc[0]
):
    raise AssertionError(
        "Customer identifier uniqueness validation failed."
    )

if (
    clv_customer_df["historical_clv"]
    .isna()
    .any()
):
    raise AssertionError(
        "Historical CLV contains missing values."
    )

if np.isinf(
    clv_customer_df["historical_clv"]
    .to_numpy(dtype=float)
).any():
    raise AssertionError(
        "Historical CLV contains infinite values."
    )

if (
    clv_customer_df["historical_clv"]
    < 0
).any():
    raise AssertionError(
        "Historical CLV contains negative values."
    )

if (
    clv_customer_df["clv_segment"]
    .isna()
    .any()
):
    raise AssertionError(
        "CLV segment contains missing values."
    )

print("All customer-level CLV assertions passed.")

All customer-level CLV assertions passed.


In [37]:
customer_clv_csv_path = (
    CLV_OUTPUT_PATH
    / "customer_lifetime_value.csv"
)

clv_summary_csv_path = (
    CLV_OUTPUT_PATH
    / "clv_segment_summary.csv"
)

clv_overall_summary_csv_path = (
    CLV_OUTPUT_PATH
    / "clv_overall_summary.csv"
)

clv_validation_csv_path = (
    CLV_OUTPUT_PATH
    / "clv_validation.csv"
)

clv_customer_df.to_csv(
    customer_clv_csv_path,
    index=False
)

clv_summary_df.to_csv(
    clv_summary_csv_path,
    index=False
)

clv_overall_summary_df.to_csv(
    clv_overall_summary_csv_path,
    index=False
)

clv_validation_df.to_csv(
    clv_validation_csv_path,
    index=False
)

print("CSV files exported successfully:")
print(customer_clv_csv_path)
print(clv_summary_csv_path)
print(clv_overall_summary_csv_path)
print(clv_validation_csv_path)

CSV files exported successfully:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\customer_lifetime_value\customer_lifetime_value.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\customer_lifetime_value\clv_segment_summary.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\customer_lifetime_value\clv_overall_summary.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\processed\customer_lifetime_value\clv_validation.csv


In [38]:
customer_clv_table_name = "customer_lifetime_value"
clv_summary_table_name = "clv_segment_summary"
clv_overall_summary_table_name = "clv_overall_summary"
clv_validation_table_name = "clv_validation"

clv_customer_df.to_sql(
    customer_clv_table_name,
    engine,
    schema=OUTPUT_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=5000
)

clv_summary_df.to_sql(
    clv_summary_table_name,
    engine,
    schema=OUTPUT_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=5000
)

clv_overall_summary_df.to_sql(
    clv_overall_summary_table_name,
    engine,
    schema=OUTPUT_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=5000
)

clv_validation_df.to_sql(
    clv_validation_table_name,
    engine,
    schema=OUTPUT_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=5000
)

print("Power BI-ready PostgreSQL tables created successfully.")

Power BI-ready PostgreSQL tables created successfully.


In [39]:
created_tables_query = text(
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema = :schema_name
      AND table_name IN (
          :customer_clv_table,
          :summary_table,
          :overall_summary_table,
          :validation_table
      )
    ORDER BY table_name;
    """
)

created_tables_df = pd.read_sql(
    created_tables_query,
    engine,
    params={
        "schema_name": OUTPUT_SCHEMA,
        "customer_clv_table": customer_clv_table_name,
        "summary_table": clv_summary_table_name,
        "overall_summary_table": clv_overall_summary_table_name,
        "validation_table": clv_validation_table_name
    }
)

expected_table_names = {
    customer_clv_table_name,
    clv_summary_table_name,
    clv_overall_summary_table_name,
    clv_validation_table_name
}

actual_table_names = set(
    created_tables_df["table_name"]
)

if not expected_table_names.issubset(
    actual_table_names
):
    missing_tables = (
        expected_table_names
        - actual_table_names
    )

    raise RuntimeError(
        f"Expected CLV tables were not created: {missing_tables}"
    )

display(created_tables_df)

,table_schema,table_name
0,analytics,clv_overall_summary
1,analytics,clv_segment_summary
2,analytics,clv_validation
3,analytics,customer_lifetime_value


In [40]:
customer_clv_columns_query = text(
    """
    SELECT
        column_name,
        data_type,
        udt_name
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY ordinal_position;
    """
)

customer_clv_columns_df = pd.read_sql(
    customer_clv_columns_query,
    engine,
    params={
        "schema_name": OUTPUT_SCHEMA,
        "table_name": customer_clv_table_name
    }
)

expected_customer_clv_columns = set(
    clv_customer_df.columns
)

actual_customer_clv_columns = set(
    customer_clv_columns_df["column_name"]
)

if not expected_customer_clv_columns.issubset(
    actual_customer_clv_columns
):
    missing_columns = (
        expected_customer_clv_columns
        - actual_customer_clv_columns
    )

    raise RuntimeError(
        f"CLV table is missing expected columns: {missing_columns}"
    )

display(customer_clv_columns_df)

,column_name,data_type,udt_name
0,customer_id,text,text
1,historical_customer_value,double precision,float8
2,customer_lifetime_duration,double precision,float8
3,historical_clv,double precision,float8
4,estimated_future_clv,double precision,float8
5,clv_segment,text,text


In [41]:
database_row_count_query = text(
    f"""
    SELECT COUNT(*) AS row_count
    FROM "{OUTPUT_SCHEMA}"."{customer_clv_table_name}";
    """
)

database_row_count_df = pd.read_sql(
    database_row_count_query,
    engine
)

database_row_count = int(
    database_row_count_df.loc[
        0,
        "row_count"
    ]
)

python_row_count = len(
    clv_customer_df
)

if database_row_count != python_row_count:
    raise RuntimeError(
        "PostgreSQL row count does not match the Python CLV output."
    )

print(
    f"Python CLV rows: {python_row_count:,}"
)

print(
    f"PostgreSQL CLV rows: {database_row_count:,}"
)

print(
    "Python and PostgreSQL row counts match."
)

Python CLV rows: 99,441
PostgreSQL CLV rows: 99,441
Python and PostgreSQL row counts match.


In [42]:
final_output_files = sorted(
    CLV_OUTPUT_PATH.glob("*.csv")
)

print("Final Power BI-ready CSV files:")

for file_path in final_output_files:
    print(f"- {file_path.name}")

print("\nFinal PostgreSQL tables:")

for table_name in sorted(
    expected_table_names
):
    print(
        f"- {OUTPUT_SCHEMA}.{table_name}"
    )

print("\nCustomer Lifetime Value notebook completed successfully.")

Final Power BI-ready CSV files:
- clv_overall_summary.csv
- clv_segment_summary.csv
- clv_validation.csv
- customer_lifetime_value.csv

Final PostgreSQL tables:
- analytics.clv_overall_summary
- analytics.clv_segment_summary
- analytics.clv_validation
- analytics.customer_lifetime_value

Customer Lifetime Value notebook completed successfully.
